In [10]:
import mido

class MIDITokenizer:
    def __init__(self):
        # Token ranges
        self.note_on_base = 0
        self.note_off_base = 128
        self.time_shift_base = 256
        self.velocity_base = 356

        self.time_shift_bins = 100 # up to 1 second (100 * 10ms)
        self.velocity_bins = 32

        self.pad_id = 388
        self.sos_id = 389
        self.eos_id = 390

        self.vocab_size = 391

    # ---------------- ENCODE ---------------- #
    def midi_to_events(self, midi_path: str) -> list[int]:
        mid = mido.MidiFile(midi_path)
        events = []

        all_msgs = []
        for track in mid.tracks:
            time = 0
            for msg in track:
                time += msg.time
                all_msgs.append((time, msg))

        all_msgs.sort(key=lambda x: x[0])
        last_time = 0

        for time, msg in all_msgs:
            delta_ticks = time - last_time
            delta_sec = mido.tick2second(delta_ticks, mid.ticks_per_beat, 500000)
            delta_ms = int(delta_sec * 100)

            # Time shifts
            while delta_ms > 0:
                shift = min(delta_ms, self.time_shift_bins)
                events.append(self.time_shift_base + shift)
                delta_ms -= shift

            # Notes
            if msg.type == "note_on" and msg.velocity > 0:
                vel_bin = min(msg.velocity // 4, 31)
                events.append(self.velocity_base + vel_bin)
                events.append(self.note_on_base + msg.note)

            elif msg.type == "note_off" or (msg.type == "note_on" and msg.velocity == 0):
                events.append(self.note_off_base + msg.note)

            last_time = time

        return events

    # ---------------- DECODE ---------------- #
    def events_to_midi(self, events: list[int], out_path="out.mid"):
        mid = mido.MidiFile()
        track = mido.MidiTrack()
        mid.tracks.append(track)

        current_time = 0
        current_velocity = 64

        for tok in events:
            if tok in (self.pad_id, self.sos_id, self.eos_id):
                continue

            # Note On
            if 0 <= tok < 128:
                track.append(
                    mido.Message(
                        "note_on",
                        note=tok,
                        velocity=current_velocity,
                        time=current_time,
                    )
                )
                current_time = 0

            # Note Off
            elif 128 <= tok < 256:
                track.append(
                    mido.Message(
                        "note_off",
                        note=tok - 128,
                        velocity=0,
                        time=current_time,
                    )
                )
                current_time = 0

            # Time Shift
            elif 256 <= tok < 356:
                shift = tok - 256
                delta_sec = shift * 0.01
                delta_ticks = int(mido.second2tick(delta_sec, mid.ticks_per_beat, 500000))
                current_time += delta_ticks

            # Velocity
            elif 356 <= tok < 388:
                vel_bin = tok - 356
                current_velocity = int(vel_bin * 4)

        mid.save(out_path)
        return out_path
    
tokenizer = MIDITokenizer()


In [15]:
from torch.utils.data import Dataset
import torch

class MusicDataset(Dataset):
    def __init__(self, event_list: list[int], seq_len: int, tokenizer: MIDITokenizer):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.data = [event_list[i : i + seq_len - 1] for i in range(0, len(event_list), seq_len)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx]
        
        # Input: [SOS, tokens...] + Padding
        # Label: [tokens..., EOS] + Padding
        input_seq = [self.tokenizer.sos_id] + tokens
        label_seq = tokens + [self.tokenizer.eos_id]

        # Add Padding
        padding_len = self.seq_len - len(input_seq)
        input_seq += [self.tokenizer.pad_id] * padding_len
        label_seq += [self.tokenizer.pad_id] * padding_len

        return {
            "input": torch.tensor(input_seq[:self.seq_len], dtype=torch.long),
            "label": torch.tensor(label_seq[:self.seq_len], dtype=torch.long)
        }
    
events = tokenizer.midi_to_events(r"./reverie.mid")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from model4 import MusicTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MusicTransformer(
    vocab_size=391, # 388 events + SOS, EOS, PAD
    d_model=512, 
    num_heads=8, 
    depth=6, 
    dropout=0.1
).to(device)

# We ignore the PAD token (388) in the loss calculation
criterion = nn.CrossEntropyLoss(ignore_index=388)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

dataset = MusicDataset(events, seq_len=200, tokenizer=tokenizer)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Training on {device}...")
model.train()

for epoch in range(150):
    total_loss = 0
    for batch in train_loader:
        inputs = batch["input"].to(device) # (B, seq_len)
        labels = batch["label"].to(device) # (B, seq_len)

        optimizer.zero_grad()
        
        # Forward pass: Output is (B, seq_len, vocab_size)
        logits = model(inputs)
        
        # CrossEntropy expects (B, C, L) so we transpose
        loss = criterion(logits.transpose(1, 2), labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/100 | Loss: {total_loss/len(train_loader):.4f}")

Training on cpu...


KeyboardInterrupt: 

In [9]:
import torch

def save_model(model, path="music_transformer_overfit.pt"):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "vocab_size": model.vocab_size,
        "d_model": model.d_model,
        "max_len": 200
    }
    torch.save(checkpoint, path)
    print(f"Model saved to {path}")

save_model(model)

NameError: name 'model' is not defined

In [ ]:
import torch
from model4 import MusicTransformer

def load_model(path, device):
    checkpoint = torch.load(path, map_location=device)
    
    # Re-initialize the model with the saved settings
    model = MusicTransformer(
        vocab_size=checkpoint["vocab_size"],
        d_model=checkpoint["d_model"]
    )
    
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model

model = load_model(r"./music_transformer_overfit.pt", device="cpu")

KeyError: 'max_len'

In [ ]:
import torch
import torch.nn.functional as F


def generate_music(
    model,
    tokenizer,
    max_len=2000,
    top_k=20,
    temperature=1.0,
    device="cuda",
):
    model.eval()

    tokens = [tokenizer.sos_id]

    for _ in range(max_len):
        x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)[:, -1, :] / temperature

        # Top-k filtering
        if top_k is not None:
            values, indices = torch.topk(logits, top_k)
            probs = torch.zeros_like(logits).scatter_(1, indices, values)
            probs = F.softmax(probs, dim=-1)
        else:
            probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, 1).item()

        if next_token == tokenizer.eos_id:
            break

        tokens.append(next_token)

    return tokens

generated_tokens = generate_music(
    model,
    tokenizer,
    max_len=3000,
    top_k=10,
    temperature=0.1,
    device="cpu",
)

print("Generated tokens:", generated_tokens[:50])
tokenizer.events_to_midi(generated_tokens, "out.mid")


Generated tokens: [389, 181, 279, 372, 61, 185, 281, 373, 65, 257, 189, 279, 372, 69, 258, 193, 278, 372, 43, 213, 201, 257, 197, 278, 372, 50, 171, 280, 373, 74, 373, 69, 371, 53, 178, 281, 371, 57, 257, 181, 279, 372, 43, 213, 201, 257, 190, 279, 378, 77]


'out.mid'

In [21]:
import matplotlib.pyplot as plt

# Grab the scores from the last forward pass
# Shape: (batch, num_heads, seq_len, seq_len)
scores = model.decoder_blocks[0].self_attn.attention_scores[0, 0].detach().cpu().numpy()

plt.imshow(scores, cmap='hot', interpolation='nearest')
plt.title("Attention Map: What the Model Sees")
plt.xlabel("Key (Past)")
plt.ylabel("Query (Present)")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'